In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================
# 1. Wczytanie i Przygotowanie Danych
# ==========================
print("--- 1. Przygotowywanie danych i trenowanie modelu ---")

PATH = "data_ready_for_system.csv"

# Generowanie danych syntetycznych jeśli plik nie istnieje (dla celów testowych)
if os.path.exists(PATH):
    df = pd.read_csv(PATH)
else:
    N = 600
    rng = np.random.default_rng(7)
    
    df = pd.DataFrame({
        "PH_morning": rng.uniform(5.5, 7.5, size=N),
        "PH_evening": rng.uniform(5.5, 7.5, size=N),
        "Gym": rng.choice([0, 1], size=N),
        "Liquid": rng.uniform(1.0, 4.0, size=N),
        "Lemon water": rng.choice([0, 1], size=N),
        "Water": rng.choice([0, 1], size=N),
        "Meat": rng.choice([0, 1], size=N),
        "Chocolate": rng.choice([0, 1], size=N),
        "sex": rng.choice(["F", "M"], size=N),
        "Date_01.02.2018": rng.choice([0, 1], size=N),
        "Date_01.03.2018": rng.choice([0, 1], size=N),
        "is_safe_zone": rng.choice([0, 1], size=N),
        "target": rng.choice([0, 1], size=N)
    })

TARGET = "target"

# Usuń kolumnę celu z cech
X = df.drop(columns=[TARGET])

# Usuń potencjalne kolumny pomocnicze i daty jeśli istnieją (zapobieganie wyciekowi danych)
drop_cols = ["is_safe_zone"] + [col for col in X.columns if col.startswith("Date_")]
for col in drop_cols:
    if col in X.columns:
        X = X.drop(columns=[col])

y = df[TARGET]

# ==========================
# 2. Klasyfikacja czy regresja?
# ==========================
unique_values = y.nunique()

is_classification = (
    unique_values <= 10
    and np.all(np.isclose(y, y.astype(int)))
)

print(f"Liczba unikalnych wartości target: {unique_values}")
print(f"Typ problemu: {'KLASYFIKACJA' if is_classification else 'REGRESJA'}")

# ==========================
# 3. Podział danych
# ==========================
if is_classification:
    class_counts = y.value_counts()
    if class_counts.min() >= 2:
        stratify = y
    else:
        stratify = None
        print("UWAGA: Niektóre klasy mają mniej niż 2 próbki. Wyłączam stratify.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=stratify
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

# ==========================
# 4. Preprocessing ( pipeline )
# ==========================
num_cols = X_train.select_dtypes(include=np.number).columns
cat_cols = X_train.select_dtypes(exclude=np.number).columns

preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            cat_cols,
        )
    ],
    remainder="drop",
)

# ==========================
# 5. Budowa i Trenowanie Modelu
# ==========================
proba = None  # Inicjalizacja, aby uniknąć błędu "possibly unbound"

if is_classification:
    model = Pipeline([
        ("pre", preprocessor),
        ("model", LogisticRegression(max_iter=3000)),
    ])
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    
    # Obsługa binarnej vs wieloklasowej klasyfikacji dla ROC AUC
    if unique_values == 2:
        proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, proba)
    else:
        proba = model.predict_proba(X_test)
        auc = roc_auc_score(y_test, proba, multi_class="ovr")

    print("\n=== WYNIKI KLASYFIKACJI ===")
    print("ACC:", round(accuracy_score(y_test, pred), 4))
    print("AUC:", round(auc, 4))
else:
    model = Pipeline([
        ("pre", preprocessor),
        ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ])
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    print("\n=== WYNIKI REGRESJI ===")
    print("MAE:", round(mean_absolute_error(y_test, pred), 4))
    print("R²:", round(r2_score(y_test, pred), 4))

# ==========================
# 6. Ewaluacja i Istotność Cech
# ==========================
print("\n--- 2. Ewaluacja i analiza dopasowania ---")

try:
    feature_names = model.named_steps["pre"].get_feature_names_out()
except Exception:
    feature_names = num_cols.tolist() + cat_cols.tolist()

print(f"Liczba cech wejściowych do modelu: {len(feature_names)}")

if is_classification:
    coefficients = model.named_steps["model"].coef_[0]
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance/Coefficient": coefficients
    }).sort_values(by="Importance/Coefficient", key=abs, ascending=False)
    
    print("\nTop 10 najsilniejszych cech (współczynniki):")
    print(importance_df.head(10).to_string(index=False))
else:
    importances = model.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)
    
    print("\nTop 10 najważniejszych cech (Feature Importance):")
    print(importance_df.head(10).to_string(index=False))

# ==========================
# 7. Zapisywanie Prognoz do Pliku
# ==========================
print("\n--- 3. Generowanie i zapisywanie wyników końcowych ---")

output_df = X_test.copy()
output_df["True_Target"] = y_test
output_df["Predicted_Target"] = pred

if is_classification and proba is not None:
    if unique_values == 2:
        output_df["Confidence_Score"] = proba
    else:
        output_df["Confidence_Score"] = proba.max(axis=1)
else:
    output_df["Absolute_Error"] = np.abs(y_test - pred)

output_path = "model_predictions_output.csv"
output_df.to_csv(output_path, index=False)
print(f"Sukces! Wyniki i prognozy zostały zapisane do pliku: '{output_path}'")

--- 1. Przygotowywanie danych i trenowanie modelu ---
Liczba unikalnych wartości target: 124
Typ problemu: REGRESJA

=== WYNIKI REGRESJI ===
MAE: 0.0708
R²: 0.962

--- 2. Ewaluacja i analiza dopasowania ---
Liczba cech wejściowych do modelu: 97

Top 10 najważniejszych cech (Feature Importance):
             Feature  Importance
     num__PH_evening    0.346185
     num__PH_morning    0.316849
      num__PH_midday    0.285302
         num__Liquid    0.004552
          num__Water    0.004032
          num__Apple    0.003653
num__Vegetable salad    0.002940
      num__Fruit tea    0.002741
          num__Bread    0.002662
         num__Cookie    0.002440

--- 3. Generowanie i zapisywanie wyników końcowych ---
Sukces! Wyniki i prognozy zostały zapisane do pliku: 'model_predictions_output.csv'


In [3]:
# ==========================
# 8. Wyjaśnialna AI (XAI)
# ==========================
print("\n--- 4. Obliczanie wskaźników XAI (PFI, PDP, ICE, SHAP, LIME) ---")

import lime
import lime.lime_tabular
import shap
from sklearn.inspection import partial_dependence, permutation_importance

# Przygotowanie transformowanych danych dla SHAP i LIME (model operuje na danych po preprocesorze)
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# Pobranie poprawnych nazw cech po One-Hot Encodingu
try:
    feature_names_trans = preprocessor.get_feature_names_out()
except Exception:
    feature_names_trans = [f"f_{i}" for i in range(X_train_trans.shape[1])]

# Pobranie samego wewnętrznego modelu (LogisticRegression lub RandomForestRegressor)
trained_model = model.named_steps["model"]

# --- A. Permutation Feature Importance (PFI) ---
print("\n[PFI] Obliczanie Permutation Feature Importance...")
# Dla klasyfikacji używamy accuracy, dla regresji r2
scoring_metric = "accuracy" if is_classification else "r2"
pfi_results = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42, scoring=scoring_metric
)

pfi_df = pd.DataFrame(
    {
        "Feature": X_test.columns,
        "Importance_Mean": pfi_results.importances_mean,
        "Importance_Std": pfi_results.importances_std,
    }
).sort_values(by="Importance_Mean", ascending=False)

print("\nTop 5 cech według PFI:")
print(pfi_df.head(5).to_string(index=False))


# --- B. Partial Dependence Plot (PDP) & Individual Conditional Expectation (ICE) ---
print("\n[PDP/ICE] Generowanie krzywych zależności dla najważniejszej cechy...")
# Wybieramy najważniejszą numeryczną cechę na podstawie analizy danych
top_num_feature = num_cols[0] if len(num_cols) > 0 else X_test.columns[0]

pdp_ice = partial_dependence(
    model, X_test, features=[top_num_feature], kind="both", grid_resolution=20
)

print(f"-> Skrajne wartości PDP dla cechy '{top_num_feature}':")
print(
    f"   Wartości cechy: {pdp_ice['grid_values'][0][:3]} ... {pdp_ice['grid_values'][0][-3:]}"
)
print(
    f"   Średnia odpowiedź modelu (PDP): {pdp_ice['average'][0][:3]} ... {pdp_ice['average'][0][-3:]}"
)
print(f"   Liczba linii indywidualnych (ICE): {len(pdp_ice['individual'][0])}")


# --- C. SHAP (SHapley Additive exPlanations) ---
print("\n[SHAP] Obliczanie wartości Shapleya...")
# Używamy LinearExplainer dla regresji logistycznej lub TreeExplainer dla Random Forest
if is_classification:
    explainer_shap = shap.LinearExplainer(trained_model, X_train_trans)
    shap_values = explainer_shap.shap_values(X_test_trans)
else:
    explainer_shap = shap.TreeExplainer(trained_model)
    shap_values = explainer_shap.shap_values(X_test_trans)

# Wyciągamy średni absolutny wpływ cechy na predykcję
if isinstance(
    shap_values, list
):  # Obsługa reprezentacji wieloklasowej / binarnej w SHAP
    mean_shap = np.abs(shap_values[0]).mean(axis=0)
else:
    mean_shap = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame(
    {"Feature": feature_names_trans, "Mean_SHAP": mean_shap}
).sort_values(by="Mean_SHAP", ascending=False)

print("\nTop 5 cech według SHAP:")
print(shap_df.head(5).to_string(index=False))


# --- D. LIME (Local Interpretable Model-agnostic Explanations) ---
print("\n[LIME] Wyjaśnianie pojedynczej prognozy...")
mode_lime = "classification" if is_classification else "regression"

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_trans,
    feature_names=feature_names_trans,
    class_names=["Class_0", "Class_1"] if is_classification else None,
    mode=mode_lime,
    random_state=42,
)

# Wyjaśniamy pierwszą próbkę ze zbioru testowego
idx = 0
if is_classification:
    exp = explainer_lime.explain_instance(
        X_test_trans[idx], trained_model.predict_proba, num_features=5
    )
else:
    exp = explainer_lime.explain_instance(
        X_test_trans[idx], trained_model.predict, num_features=5
    )

print(f"\nLokalne wyjaśnienie LIME dla obserwacji o indeksie {idx}:")
for feature_interaction, weight in exp.as_list():
    print(f"  {feature_interaction:<40} | Waga LIME: {weight:+.4f}")



--- 4. Obliczanie wskaźników XAI (PFI, PDP, ICE, SHAP, LIME) ---


c:\Users\Patryk\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[PFI] Obliczanie Permutation Feature Importance...


c:\Users\Patryk\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Patryk\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Patryk\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\User


Top 5 cech według PFI:
        Feature  Importance_Mean  Importance_Std
     PH_evening         0.416424        0.038419
      PH_midday         0.348885        0.046863
     PH_morning         0.332543        0.062959
          Apple         0.001866        0.000709
Vegetable salad         0.000299        0.000602

[PDP/ICE] Generowanie krzywych zależności dla najważniejszej cechy...
-> Skrajne wartości PDP dla cechy 'PH_morning':
   Wartości cechy: [-1.47913602 -1.33372239 -1.18830877] ... [0.99289561 1.13830924 1.28372286]
   Średnia odpowiedź modelu (PDP): [5.70840063 5.71401761 5.72258931] ... [6.18691447 6.20092138 6.21568742]
   Liczba linii indywidualnych (ICE): 53

[SHAP] Obliczanie wartości Shapleya...

Top 5 cech według SHAP:
        Feature  Mean_SHAP
num__PH_evening   0.184505
 num__PH_midday   0.177042
num__PH_morning   0.174309
    num__Liquid   0.002885
     num__Apple   0.002812

[LIME] Wyjaśnianie pojedynczej prognozy...

Lokalne wyjaśnienie LIME dla obserwacji o ind